#  ROL 1 – Modelado y Generación de Datos Sintéticos
## Caso SUNBURST – SolarWinds

**Curso:** Gestión de Datos  
**Facultad:** Ingeniería – Universidad de San Buenaventura  
**Marco de referencia:** DAMA DMBOK  
**Rol:** Diseñador de Datos

---

### Contexto del Caso
En diciembre de 2020, SolarWinds descubrió que su software **Plataforma Orion** había sido comprometido por un Actor de Amenazas (AA). El malware **SUNBURST** fue inyectado en el proceso de compilación del software entre octubre de 2019 y junio de 2020, afectando versiones específicas. Aunque inicialmente se estimaron **~18,000 instalaciones activadas** como afectadas, las investigaciones posteriores redujeron la cifra a **menos de 100 clientes realmente comprometidos**.

Este notebook documenta el **diseño del modelo de datos** y la **generación de datos sintéticos** que representan los sistemas de información involucrados.

## 1. Importaciones y Configuración

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import seaborn as sns
import os
import sys

# Semilla para reproducibilidad
SEED = 42
np.random.seed(SEED)
fake = Faker('es_MX')
fake.seed_instance(SEED)

# Estilo visual
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print('✔ Librerías cargadas correctamente')
print(f'  pandas  : {pd.__version__}')
print(f'  numpy   : {np.__version__}')

## 2. Modelo Entidad-Relación (ER)

### Decisiones de Diseño

El modelo fue diseñado a partir del análisis del caso SUNBURST identificando los **actores y artefactos clave** del incidente:

| Entidad | Justificación |
|---|---|
| **Clientes** | Organizaciones que usaban Orion; núcleo del impacto |
| **Versiones_Software** | La vulnerabilidad estaba en versiones específicas |
| **Instalaciones** | Relaciona qué cliente instaló qué versión y cuándo |

### Relaciones
- Un **Cliente** puede tener muchas **Instalaciones** (1:N)
- Una **Versión_Software** puede estar en muchas **Instalaciones** (1:N)
- Una **Instalación** pertenece a exactamente 1 cliente y 1 versión (N:1)

### Atributos clave seleccionados
- `criticidad` en Clientes → permite simular el impacto real del ataque (organizaciones críticas como agencias de gobierno)
- `contiene_sunburst` en Versiones → es la bandera que distingue versiones vulnerables
- `nivel_datos_sensibles` en Instalaciones → captura la exposición de datos por instalación

## 3. Generación de Datos Sintéticos

### 3.1 DataFrame 1: Clientes (50 registros)

In [ ]:
def generar_clientes(n=50):
    """
    Genera clientes ficticios representativos del universo real
    de usuarios de SolarWinds Orion (sectores gobierno, defensa,
    tecnología, finanzas, etc.).
    """
    tipos_org = ['Gobierno Federal', 'Gobierno Estatal', 'Empresa Privada',
                 'Empresa de Tecnología', 'Institución Financiera',
                 'Organización Militar', 'Universidad']
    sectores = ['Energía', 'Defensa', 'Finanzas', 'Tecnología',
                'Salud', 'Educación', 'Telecomunicaciones',
                'Manufactura', 'Retail', 'Gobierno']
    # EE.UU. concentra la mayoría de organizaciones afectadas según el caso
    paises = (['Estados Unidos'] * 30 + ['Reino Unido'] * 5 +
              ['Canadá'] * 5 + ['Alemania'] * 3 +
              ['Australia'] * 3 + ['Francia'] * 2 + ['Israel'] * 2)
    criticidades = ['CRÍTICA', 'ALTA', 'MEDIA', 'BAJA']
    pesos = [0.25, 0.35, 0.30, 0.10]

    registros = []
    for i in range(1, n + 1):
        registros.append({
            'cliente_id': f'CLI-{i:04d}',
            'nombre_organizacion': fake.company(),
            'tipo_org': np.random.choice(tipos_org),
            'pais': np.random.choice(paises),
            'sector': np.random.choice(sectores),
            'criticidad': np.random.choice(criticidades, p=pesos)
        })
    return pd.DataFrame(registros)


df_clientes = generar_clientes(50)
print(f'Forma del DataFrame: {df_clientes.shape}')
print(f'\nTipos de datos:')
print(df_clientes.dtypes)
df_clientes.head(8)

In [ ]:
# ── Análisis exploratorio de clientes ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribución por criticidad
colores = {'CRÍTICA': '#d62728', 'ALTA': '#ff7f0e', 'MEDIA': '#2ca02c', 'BAJA': '#1f77b4'}
criticos = df_clientes['criticidad'].value_counts()
axes[0].bar(criticos.index, criticos.values,
            color=[colores[c] for c in criticos.index], edgecolor='white', linewidth=1.5)
axes[0].set_title('Clientes por Nivel de Criticidad', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Número de clientes')

# Distribución por país
pais_count = df_clientes['pais'].value_counts()
axes[1].barh(pais_count.index, pais_count.values, color=sns.color_palette('Set2', len(pais_count)))
axes[1].set_title('Clientes por País', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Número de clientes')

plt.tight_layout()
plt.savefig('../visualizations/clientes_exploratorio.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ Gráfico guardado')

### 3.2 DataFrame 2: Versiones de Software (8 registros)

In [ ]:
def generar_versiones_software():
    """
    Datos basados directamente en el caso SUNBURST.
    Las versiones 2019.4 (anterior a HF5) y 2020.2 / 2020.2 HF1
    contienen el malware. Las versiones HF5 y 2020.2.1 son los parches.
    """
    versiones = [
        # VULNERABLES
        {'version_id':'VER-001','nombre_version':'Orion 2019.4',
         'fecha_release':'2019-10-15','contiene_sunburst':True,'fecha_compilacion':'2019-10-10'},
        {'version_id':'VER-002','nombre_version':'Orion 2019.4 HF1',
         'fecha_release':'2019-12-01','contiene_sunburst':True,'fecha_compilacion':'2019-11-28'},
        {'version_id':'VER-003','nombre_version':'Orion 2019.4 HF2',
         'fecha_release':'2020-01-15','contiene_sunburst':True,'fecha_compilacion':'2020-01-12'},
        {'version_id':'VER-004','nombre_version':'Orion 2020.2',
         'fecha_release':'2020-02-25','contiene_sunburst':True,'fecha_compilacion':'2020-02-20'},
        {'version_id':'VER-005','nombre_version':'Orion 2020.2 HF1',
         'fecha_release':'2020-04-10','contiene_sunburst':True,'fecha_compilacion':'2020-04-07'},
        # SEGURAS
        {'version_id':'VER-006','nombre_version':'Orion 2019.4 HF5',
         'fecha_release':'2020-03-26','contiene_sunburst':False,'fecha_compilacion':'2020-03-24'},
        {'version_id':'VER-007','nombre_version':'Orion 2020.2.1',
         'fecha_release':'2020-12-15','contiene_sunburst':False,'fecha_compilacion':'2020-12-13'},
        {'version_id':'VER-008','nombre_version':'Orion 2020.2.1 HF2',
         'fecha_release':'2021-01-11','contiene_sunburst':False,'fecha_compilacion':'2021-01-09'},
    ]
    df = pd.DataFrame(versiones)
    df['fecha_release'] = pd.to_datetime(df['fecha_release'])
    df['fecha_compilacion'] = pd.to_datetime(df['fecha_compilacion'])
    return df


df_versiones = generar_versiones_software()
print(f'Forma del DataFrame: {df_versiones.shape}')
print(f'\nVersiones VULNERABLES: {df_versiones[df_versiones.contiene_sunburst].shape[0]}')
print(f'Versiones SEGURAS    : {df_versiones[~df_versiones.contiene_sunburst].shape[0]}')
df_versiones

### 3.3 DataFrame 3: Instalaciones (100 registros)

In [ ]:
def generar_instalaciones(df_clientes, df_versiones, n=100):
    """
    Genera registros de instalaciones con integridad referencial garantizada:
    - cliente_id referencia a df_clientes
    - version_id referencia a df_versiones
    - Las fechas de instalación son coherentes con el release de la versión
    """
    ids_clientes = df_clientes['cliente_id'].tolist()
    ids_versiones = df_versiones['version_id'].tolist()
    # Más peso a versiones vulnerables (refleja realidad del caso)
    pesos_version = [0.15, 0.15, 0.10, 0.20, 0.10, 0.10, 0.12, 0.08]
    niveles = ['BAJO', 'MEDIO', 'ALTO', 'MUY_ALTO']
    pesos_nivel = [0.30, 0.40, 0.20, 0.10]

    registros = []
    for i in range(1, n + 1):
        version_id = np.random.choice(ids_versiones, p=pesos_version)
        fecha_release = df_versiones.loc[
            df_versiones['version_id'] == version_id, 'fecha_release'
        ].values[0]
        fecha_dt = pd.Timestamp(fecha_release)
        dias = np.random.randint(0, 540)
        fecha_inst = min(fecha_dt + pd.Timedelta(days=int(dias)), pd.Timestamp('2021-05-07'))

        registros.append({
            'instalacion_id': f'INS-{i:04d}',
            'cliente_id': np.random.choice(ids_clientes),
            'version_id': version_id,
            'fecha_instalacion': fecha_inst.date(),
            'nivel_datos_sensibles': np.random.choice(niveles, p=pesos_nivel)
        })
    return pd.DataFrame(registros)


df_instalaciones = generar_instalaciones(df_clientes, df_versiones, 100)
print(f'Forma del DataFrame: {df_instalaciones.shape}')
print(f'\nInstalaciones con versión VULNERABLE: '
      f"{df_instalaciones[df_instalaciones.version_id.isin(df_versiones[df_versiones.contiene_sunburst].version_id)].shape[0]}")
df_instalaciones.head(8)

## 4. Verificación de Integridad Referencial

In [ ]:
# Verificar que todos los cliente_id en instalaciones existen en clientes
ids_clientes_validos = set(df_clientes['cliente_id'])
ids_versiones_validos = set(df_versiones['version_id'])

clientes_huerfanos = df_instalaciones[
    ~df_instalaciones['cliente_id'].isin(ids_clientes_validos)
]
versiones_huerfanas = df_instalaciones[
    ~df_instalaciones['version_id'].isin(ids_versiones_validos)
]

print('── Verificación de Integridad Referencial ──')
print(f'  cliente_id sin referencia en Clientes      : {len(clientes_huerfanos)}')
print(f'  version_id sin referencia en Versiones     : {len(versiones_huerfanas)}')

if len(clientes_huerfanos) == 0 and len(versiones_huerfanas) == 0:
    print('\n  ✔ INTEGRIDAD REFERENCIAL: OK – sin registros huérfanos')
else:
    print('\n  ✘ Se encontraron violaciones de integridad referencial')

## 5. Visualizaciones de los Datos Generados

In [ ]:
# ── Visualización 1: Instalaciones por versión (vulnerable vs segura) ────────
ins_merged = df_instalaciones.merge(
    df_versiones[['version_id', 'nombre_version', 'contiene_sunburst']],
    on='version_id'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras: instalaciones por versión
conteo = ins_merged.groupby(['nombre_version', 'contiene_sunburst']).size().reset_index(name='count')
colores_vuln = {True: '#d62728', False: '#2ca02c'}
bars = axes[0].bar(
    conteo['nombre_version'],
    conteo['count'],
    color=[colores_vuln[v] for v in conteo['contiene_sunburst']],
    edgecolor='white'
)
axes[0].set_title('Instalaciones por Versión de Software', fontweight='bold')
axes[0].set_ylabel('Número de instalaciones')
axes[0].tick_params(axis='x', rotation=45)
patch_v = mpatches.Patch(color='#d62728', label='Contiene SUNBURST')
patch_s = mpatches.Patch(color='#2ca02c', label='Versión segura')
axes[0].legend(handles=[patch_v, patch_s])

# Gráfico de pastel: proporción vulnerable vs segura
total_vuln = ins_merged[ins_merged['contiene_sunburst']].shape[0]
total_seg  = ins_merged[~ins_merged['contiene_sunburst']].shape[0]
axes[1].pie(
    [total_vuln, total_seg],
    labels=[f'Vulnerables\n({total_vuln})', f'Seguras\n({total_seg})'],
    colors=['#d62728', '#2ca02c'],
    autopct='%1.1f%%',
    startangle=90,
    explode=(0.05, 0)
)
axes[1].set_title('Proporción de Instalaciones\nVulnerables vs Seguras', fontweight='bold')

plt.suptitle('Análisis de Instalaciones – Plataforma Orion SUNBURST', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../visualizations/instalaciones_distribucion.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ Gráfico guardado')

In [ ]:
# ── Visualización 2: Línea de tiempo de versiones ───────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))

for _, row in df_versiones.iterrows():
    color = '#d62728' if row['contiene_sunburst'] else '#2ca02c'
    marker = '✗' if row['contiene_sunburst'] else '✓'
    ax.scatter(row['fecha_release'], 0.5, s=250, color=color, zorder=5)
    ax.annotate(
        f"{row['nombre_version']}\n{row['fecha_release'].strftime('%b %Y')}",
        xy=(row['fecha_release'], 0.5),
        xytext=(0, 25 if list(df_versiones.index).index(_) % 2 == 0 else -45),
        textcoords='offset points',
        ha='center', fontsize=8,
        arrowprops=dict(arrowstyle='->', color='gray', lw=1)
    )

ax.axhline(0.5, color='gray', lw=1.5, linestyle='--', alpha=0.5)
ax.set_xlim(pd.Timestamp('2019-06-01'), pd.Timestamp('2021-06-01'))
ax.set_ylim(0, 1)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
ax.set_yticks([])
ax.set_title('Línea de Tiempo: Versiones de Orion y SUNBURST', fontsize=13, fontweight='bold')
patch_v = mpatches.Patch(color='#d62728', label='Versión VULNERABLE')
patch_s = mpatches.Patch(color='#2ca02c', label='Versión SEGURA / Parche')
ax.legend(handles=[patch_v, patch_s], loc='upper left')
plt.tight_layout()
plt.savefig('../visualizations/timeline_versiones.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ Gráfico guardado')

## 6. Exportación de CSVs

In [ ]:
os.makedirs('../data', exist_ok=True)
os.makedirs('../visualizations', exist_ok=True)

df_clientes.to_csv('../data/clientes.csv', index=False, encoding='utf-8')
df_versiones.to_csv('../data/versiones_software.csv', index=False, encoding='utf-8')
df_instalaciones.to_csv('../data/instalaciones.csv', index=False, encoding='utf-8')

print('✔ Archivos CSV exportados:')
for f in ['clientes.csv', 'versiones_software.csv', 'instalaciones.csv']:
    path = f'../data/{f}'
    size = os.path.getsize(path) if os.path.exists(path) else '?'
    print(f'   ../data/{f}  ({size} bytes)')

## 7. Resumen del Modelo de Datos

| DataFrame | Registros | Columnas clave | Relaciones |
|---|---|---|---|
| `clientes` | 50 | cliente_id (PK), criticidad, sector | ← Instalaciones |
| `versiones_software` | 8 | version_id (PK), contiene_sunburst | ← Instalaciones |
| `instalaciones` | 100 | instalacion_id (PK), cliente_id (FK), version_id (FK) | → Clientes, Versiones |

### Hallazgos del Modelado
1. **~71% de las instalaciones** usan versiones vulnerables, reflejando el caso real donde la mayoría de los 18,000 downloads eran de versiones comprometidas.
2. El atributo `contiene_sunburst` en Versiones es la variable discriminante que permite simular la distinción entre **expuestos (18,000) vs comprometidos (<100)**.
3. La integridad referencial está garantizada: todos los `cliente_id` y `version_id` en Instalaciones existen en sus respectivas tablas maestras.